# Neuromaps-PRIME: a hands-on tutorial

In Neuromaps-PRIME, coordinate spaces are linked together to form a network. This notebook teaches you to move data through that network, and
then to visualize and analyze the result.

The worked example carries two macaque cortical maps from the Yerkes19 surface onto the human fsLR
surface, and asks whether they are related once they share a space.

**What you will be able to do**

1. Load the template-space graph and see what maps are available
2. Transform an annotation from one space into another, including across species
3. Visualize the transformed map
4. Compare two maps with a statistically appropriate null model
5. Add your own template space and connect it to the graph

**Setup.** You need Python and a container runtime (Docker, Podman, or Apptainer/Singularity).
You do not strictly need to have Connectome Workbench or ANTs installed. If these are not available locally, they will be used via a container runtime environemnt. Sections marked *offline* need neither downloads nor a container.

**Disclaimer:** Neuromaps-PRIME applies transformations produced by other people and takes them
at face value. Whether a given registration is anatomically correct is an empirical question this
package does not answer. The benchmark in the final section measures software self-consistency,
which is a different thing.

---
## 1. Setup

In [ ]:
!python3 -m pip install neuromaps-prime matplotlib scipy

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import tempfile

from neuromaps_prime.niwrap import resolve_runner

from scipy.spatial import ConvexHull


runner_type, executable = resolve_runner("auto")
print(f"runner: {runner_type} ({executable})")

if runner_type == "local":
    print("\nNo container runtime found. The graph, statistics, and")
    print("space-registration sections still work; transformations will not.")

`runner="auto"` prefers Docker, then Podman, then Apptainer/Singularity, falling back to local
execution. You can request one explicitly with `NeuromapsGraph(runner="docker")`, or point the
download cache somewhere specific with `data_dir=Path(...)`.

In [ ]:
from neuromaps_prime.graph import NeuromapsGraph

graph = NeuromapsGraph(verbose=1)
print(graph)
print("cache:", graph.data_dir)

---
## 2. What is in the graph *(offline)*

Nodes are template spaces; edges are transformations between them.

In [ ]:
TUTORIAL_SPECIES = ("human", "macaque")

for species, node in sorted(
    (graph.get_node_data(name).species, name)
    for name in graph.nodes(data=False)
    if graph.get_node_data(name).species in TUTORIAL_SPECIES
):
    print(f"{node:14s}\t{species}")

This tutorial uses the human and macaque spaces, which form the most densely connected part of the
graph and have the richest set of curated transformations. The graph also contains spaces for other
species; some are experimental or not yet connected in both directions, so they are left aside here.

Each space lists the annotations defined on it, and inspecting that list is the natural starting
point: it shows which maps exist, at which vertex densities, and for which hemispheres.

In [ ]:
node = graph.get_node_data("Yerkes19")

print(f"{node.description}\n")
for a in sorted(node.surface_annotations, key=lambda x: (x.label, x.density)):
    print(f"  {a.label:20s} {a.density:6s} {a.hemisphere}")

The labels are short type codes, optionally suffixed to distinguish variants:

| Code | Annotation Type    | 
|------|--------------------|
| `CT` | cortical thickness |
| `CV` | curvature | 
| `MM` | myelin map | 
| `SMM`| smoothed myelin |
| `PC_*` | parcellations | 
| `RM_*` | receptor maps | 
| `SD` | sulcal depth | 
| `BM` | brain mask |

**Coverage is uneven.** Notice that `MM` is defined only for the right hemisphere, while `CT` and
`SMM` have both. Checking before you request saves a confusing failure later. This is why we use the
right hemisphere throughout the tutorial.

---
## 3. How transformations are found *(offline)*

You will rarely call this directly, but it explains what the transformer does internally. Given a
source and target space, the package searches for the shortest path through the graph and applies
each transformation along it in turn.

In [ ]:
print(
    "Yerkes19 -> fsLR :",
    graph.find_path("Yerkes19", "fsLR", edge_type="surface_to_surface"),
)
print(
    "MEBRAINS -> fsLR :",
    graph.find_path("MEBRAINS", "fsLR", edge_type="surface_to_surface"),
)

The first is a direct edge. The second has no direct transformation, yet the request still succeeds:
the package composes a within-species macaque registration with a cross-species macaque-to-human
registration. This is what the graph representation buys you, and it is why adding one registration
can yield transformations to many spaces.

Two things are worth knowing about the result:

- **An empty list means no route exists**, which is information rather than an error. Asking for a
  volumetric route between species returns nothing, because no cross-species volumetric
  transformations are currently curated.
- **The graph is directed.** Not every transformation has been declared in both directions, so a
  route may exist one way and not the other. Section 7 shows where that comes from.

In [ ]:
print("surface:", graph.find_path("MEBRAINS", "fsLR", edge_type="surface_to_surface"))
print("volume :", graph.find_path("MEBRAINS", "fsLR", edge_type="volume_to_volume"))

---
## 4. Transforming real data

**[needs a container and downloads]**

Now the core workflow. We fetch two macaque maps in their native Yerkes19 space and express both on
the human fsLR surface. Fetching downloads on first use and caches locally.

In [ ]:
myelin_macaque = graph.fetch_surface_annotation(
    space="Yerkes19", label="MM", density="32k", hemisphere="right"
)
thickness_macaque = graph.fetch_surface_annotation(
    space="Yerkes19", label="CT", density="32k", hemisphere="right"
)

myelin_path = myelin_macaque.fetch()
thickness_path = thickness_macaque.fetch()

print("myelin   :", myelin_path)
print("thickness:", thickness_path)

Each fetch returns an object carrying the local path to the downloaded file along with its metadata.
Pass that path straight to the transformer.

`transformer_type` distinguishes continuous data, which are interpolated, from discrete labels,
which are not. Myelin and thickness are continuous, so `metric` is correct. Interpolating a
parcellation would produce fractional indices corresponding to no anatomical region.

In [ ]:
myelin_on_fsLR = graph.surface_to_surface_transformer(
    transformer_type="metric",
    input_file=myelin_path,
    source_space="Yerkes19",
    target_space="fsLR",
    hemisphere="right",
    source_density="32k",
    target_density="32k",
    output_file_path="myelin_on_fsLR.func.gii",
)

thickness_on_fsLR = graph.surface_to_surface_transformer(
    transformer_type="metric",
    input_file=thickness_path,
    source_space="Yerkes19",
    target_space="fsLR",
    hemisphere="right",
    source_density="32k",
    target_density="32k",
    output_file_path="thickness_on_fsLR.func.gii",
)

print(f"\nOutput filepath: {myelin_on_fsLR}")  # prints the real path

You are not restricted to annotations that ship with the package. `input_file` is just a path, so
any GIFTI file already defined on the source space can be passed directly, provided its space,
hemisphere, and density match what the call declares. Registering a map as an annotation (Section 7)
makes it discoverable and reusable, but is not a precondition for transforming it.

If `source_density` is omitted it is inferred from the vertex count of the input file.

In [ ]:
before = nib.load(myelin_path).darrays[0].data
after = nib.load(myelin_on_fsLR).darrays[0].data

print(
    f"Yerkes19: {before.shape[0]:>7,} vertices  range [{before.min():.2f}, {before.max():.2f}]"
)
print(
    f"fsLR    : {after.shape[0]:>7,} vertices  range [{after.min():.2f}, {after.max():.2f}]"
)

The vertex count changes because the surfaces differ, while the value range is broadly preserved:
the map has been resampled, not rescaled.

---
## 5. Visualizing the result

Neuromaps-PRIME writes standard GIFTI files and does not render surfaces itself, so any surface
plotting library will do. The example uses a custom matplotlib function, which is not a dependency and must be installed
separately.

In [ ]:
def plot_surface(
    surface_file,
    values,
    hemisphere="right",
    title="",
    cmap="viridis",
    views=("lateral", "medial"),
    percentile=(2, 98),
):
    """Render a metric on a cortical surface. Uses only matplotlib and nibabel.

    views: any of 'lateral', 'medial', 'dorsal', 'ventral', 'anterior', 'posterior'
    """
    g = nib.load(str(surface_file))
    coords = np.asarray(g.agg_data("NIFTI_INTENT_POINTSET"), dtype=float)
    faces = np.asarray(g.agg_data("NIFTI_INTENT_TRIANGLE"))

    # A right hemisphere's lateral surface faces +x; a left hemisphere's faces -x.
    lat_azim = 0 if hemisphere.lower().startswith("r") else 180
    ANGLES = {
        "lateral": (0, lat_azim),
        "medial": (0, (lat_azim + 180) % 360),
        "dorsal": (90, 270),
        "ventral": (-90, 270),
        "anterior": (0, 90),
        "posterior": (0, 270),
    }

    values = np.asarray(values, dtype=float)
    face_vals = values[faces].mean(axis=1)
    finite = np.isfinite(face_vals)
    vmin, vmax = np.percentile(face_vals[finite], percentile)
    norm = plt.Normalize(vmin, vmax)
    colors = plt.get_cmap(cmap)(norm(np.nan_to_num(face_vals, nan=vmin)))

    fig, axes = plt.subplots(
        1, len(views), figsize=(6 * len(views), 5), subplot_kw={"projection": "3d"}
    )
    axes = np.atleast_1d(axes)
    for ax, view in zip(axes, views):
        elev, azim = ANGLES[view]
        ax.plot_trisurf(
            coords[:, 0],
            coords[:, 1],
            coords[:, 2],
            triangles=faces,
            shade=False,
            linewidth=0,
            antialiased=False,
        )
        ax.collections[0].set_facecolors(colors)
        ax.view_init(elev=elev, azim=azim)
        ax.set_proj_type("ortho")
        ax.set_box_aspect(np.ptp(coords, axis=0))
        ax.set_axis_off()
        ax.set_title(view, fontsize=11)
    if title:
        fig.suptitle(title)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=axes, shrink=0.6)
    return fig

In [ ]:
inflated_path = graph.fetch_surface_atlas(
    space="fsLR", density="32k", hemisphere="right", resource_type="inflated"
).fetch()

myelin_fsLR = nib.load(myelin_on_fsLR).darrays[0].data

plt.close()
plot_surface(
    str(inflated_path),
    myelin_fsLR,
    title="Macaque myelin, expressed on fsLR",
)
plt.show()

Look at the map before you analyse it. Gross errors, an inverted hemisphere, a map confined to one
lobe, an unmasked medial wall, are obvious on inspection and invisible in a correlation coefficient.

---
Neuromaps-PRIME inherits the statistical tools developed in the original [Neuromaps](https://github.com/netneurolab/neuromaps) repository and packages them in the neuromaps_prime.analysis submodule. 

Reference: Markello, R.D., Hansen, J.Y., Liu, ZQ. et al. neuromaps: structural and functional interpretation of brain maps. Nat Methods 19, 1472–1479 (2022). https://doi.org/10.1038/s41592-022-01625-w

## 6. Analyzing: comparing two maps

Two maps in the same space at the same density are two vectors of equal length, so they can be
correlated. The correlation is the easy part; deciding whether it means anything is where brain maps
differ from ordinary data with independent sampling.

In [ ]:
from neuromaps_prime.analysis import compare_images

myelin = nib.load(myelin_on_fsLR).darrays[0].data
thickness = nib.load(thickness_on_fsLR).darrays[0].data

result = compare_images(myelin, thickness, metric="pearsonr")
print(f"r = {result.similarity:.3f}")
print(f"p = {result.pvalue}")

`compare_images` masks out zero and non-finite vertices, which covers the medial wall where neither
map is defined. The p-value comes back as `nan`, deliberately: the function will not give you one
until you say what the null hypothesis looks like.

### Why the default is `nan`

Brain maps are spatially autocorrelated, so neighbouring vertices are not independent observations.
A test that shuffles vertices at random destroys that structure and produces a null distribution far
narrower than reality, against which almost any smooth map looks significant.

The demonstration below uses synthetic data with known ground truth, and runs **offline**. It builds
two smooth maps on a sphere *independently*, so there is no true relationship between them.

In [ ]:
def make_demo_sphere(n=800):
    """Build a small spherical surface and save it as GIFTI."""
    i = np.arange(n) + 0.5
    phi = np.arccos(1 - 2 * i / n)
    theta = np.pi * (1 + 5**0.5) * i
    xyz = np.c_[
        np.cos(theta) * np.sin(phi), np.sin(theta) * np.sin(phi), np.cos(phi)
    ].astype(np.float32)
    tri = ConvexHull(xyz).simplices.astype(np.int32)

    img = nib.GiftiImage()
    img.add_gifti_data_array(
        nib.gifti.GiftiDataArray(xyz, intent="NIFTI_INTENT_POINTSET")
    )
    img.add_gifti_data_array(
        nib.gifti.GiftiDataArray(tri, intent="NIFTI_INTENT_TRIANGLE")
    )
    tmp = tempfile.NamedTemporaryFile(suffix=".surf.gii", delete=False)
    nib.save(img, tmp)
    return tmp.name, xyz


rng = np.random.default_rng(0)
demo_sphere, xyz = make_demo_sphere()

map_a = xyz[:, 2] + 0.4 * xyz[:, 0] + rng.normal(0, 0.15, len(xyz))
map_b = xyz[:, 0] + 0.4 * xyz[:, 1] + rng.normal(0, 0.15, len(xyz))
print(f"{len(map_a)} vertices, two independently generated smooth maps")

In [ ]:
from neuromaps_prime.analysis import permtest_metric

naive = permtest_metric(map_a, map_b, metric="pearsonr", n_perm=1000, seed=0)
print(f"r                   = {naive.similarity:.3f}")
print(f"naive permutation p = {naive.pvalue:.4f}")

We built these maps independently, so we know there is no relationship. The naive test reports a
p-value most papers would call highly significant.

### Spatial null models

The remedy is to compare the observed correlation against surrogate maps that preserve the spatial
autocorrelation of the original but destroy any true correspondence. Neuromaps-PRIME ships the
family of null models from the original Neuromaps toolbox. The classic approach rotates the map on
the sphere.

In [ ]:
from neuromaps_prime.analysis.surfaces.nulls.nulls import alexander_bloch

nulls = alexander_bloch(map_a, surface=demo_sphere, n_perm=1000, seed=0)
spin = compare_images(map_a, map_b, metric="pearsonr", nulls=nulls)

print(f"r                   = {spin.similarity:.3f}")
print(f"naive permutation p = {naive.pvalue:.4f}")
print(f"spin test p         = {spin.pvalue:.4f}")
print()
print(
    f"the naive p-value is roughly {spin.pvalue / max(naive.pvalue, 1e-9):.0f}x too small"
)

The spin test gives the answer we know to be correct. A correlation near 0.3 is simply what two
arbitrary smooth maps on a sphere look like.

**Two mistakes that produce silently wrong answers.** Neither raises an error:

1. **The nulls belong to the first map.** Internally the null array replaces the *first* argument to
   `compare_images`, so nulls must be surrogates of `map_a`, not `map_b`.
2. **Generate nulls before masking.** `compare_images` masks the null array the same way it masks the
   data, so nulls must be full length. Masking yourself first and then passing nulls misaligns them.

### Exercise 1 — Swap the nulls

Generate spins of `map_b` instead of `map_a` and pass them to `compare_images`. Does it raise an
error? Does the p-value change?

In [ ]:
# YOUR CODE HERE
# nulls_b = alexander_bloch(...)
# wrong = compare_images(map_a, map_b, metric="pearsonr", nulls=nulls_b)

In [ ]:
# Solution
nulls_b = alexander_bloch(map_b, surface=demo_sphere, n_perm=1000, seed=0)
wrong = compare_images(map_a, map_b, metric="pearsonr", nulls=nulls_b)

print(f"nulls from map_a (correct): p = {spin.pvalue:.4f}")
print(f"nulls from map_b (wrong)  : p = {wrong.pvalue:.4f}")
print()
print("No error either way. Here both happen to be non-significant, so the")
print("mistake is invisible. With real data it need not be.")

### Exercise 2 — Choose a null model

Several null models are available in `neuromaps_prime.analysis.surfaces.nulls`, and they differ in
what data they accept:

| Function | Approach | Data level |
|---|---|---|
| `alexander_bloch` | rotates the map on the sphere | **vertex** (parcellation optional) |
| `burt2018` | variogram-matched surrogates, no rotation | **vertex** (parcellation optional) |
| `vasa` | rotation, each value reassigned once | parcellated (**required**) |
| `hungarian` | rotation with optimal reassignment | parcellated (**required**) |
| `baum` | rotation, parcels reassigned by majority | parcellated (**required**) |
| `cornblath` | vertex-level spins, then re-parcellated | parcellated (**required**) |

Our maps are vertex-level, so only the first two apply. Run `burt2018` and compare it against the
spin test.

In [ ]:
# YOUR CODE HERE
# from neuromaps_prime.analysis.surfaces.nulls import burt2018
# nulls_burt = burt2018(...)

In [ ]:
# Solution
from neuromaps_prime.analysis.surfaces.nulls import burt2018, vasa

nulls_burt = burt2018(map_a, surface=demo_sphere, n_perm=1000, seed=0)
res_burt = compare_images(map_a, map_b, metric="pearsonr", nulls=nulls_burt)

print(f"alexander_bloch p = {spin.pvalue:.4f}")
print(f"burt2018        p = {res_burt.pvalue:.4f}")

try:
    vasa(map_a, surface=demo_sphere, n_perm=10, seed=0)
except TypeError as e:
    print(f"\nvasa on vertex data -> TypeError: {e}")

The two p-values differ because the models encode different notions of what "no relationship" means.
Choose on principled grounds, state the choice, and do not pick post hoc by which gives the smaller
p-value. Reporting which null model was used matters as much as reporting the correlation.

### Applying this to the real maps

With the transformed maps from Section 4, the same three calls apply. The sphere comes from the
graph rather than being synthesised:

In [ ]:
sphere = graph.fetch_surface_atlas(
    space="fsLR", density="32k", hemisphere="right", resource_type="sphere"
)

real_nulls = alexander_bloch(myelin, surface=sphere.file_path, n_perm=1000, seed=0)
real = compare_images(myelin, thickness, metric="pearsonr", nulls=real_nulls)

print(
    f"\nmyelin vs thickness on fsLR: r = {real.similarity:.3f}, p_spin = {real.pvalue:.4g}"
)

So far we've taken two continuous maps that began in a macaque coordinate system, resampled them into a human surface, and compared them against an appropriate null model.


### Comparing regional averages
We can do the same sort of comparison with data that's been binned into discrete regions of interest. 

In [ ]:
from neuromaps_prime.analysis.parcels import parcel_reduce


# The Yeo parcellation used below for demonstrative purposes is defined on the
# macaque Yerkes19 surface, so it has to be
# carried into fsLR first. Discrete labels must be interpolated with the nearest neighbor method,
#  hence transformer_type="label". Note the right hemisphere is only available at 10k,
# so the transform changes density as well as space.
atlas = graph.fetch_surface_annotation(
    space="Yerkes19", label="PC_Yeo7Networks", density="10k", hemisphere="right"
)

atlas_on_fsLR = graph.surface_to_surface_transformer(
    transformer_type="label",
    input_file=atlas.fetch(),
    source_space="Yerkes19",
    target_space="fsLR",
    hemisphere="right",
    source_density="10k",
    target_density="32k",
    output_file_path="Yeo7Networks_on_fsLR.label.gii",
)

inflated_path = graph.fetch_surface_atlas(
    space="fsLR", density="32k", hemisphere="right", resource_type="inflated"
).fetch()

atlas_values = nib.load(atlas_on_fsLR).darrays[0].data

plot_surface(
    str(inflated_path),
    atlas_values,
    title="Macaque myelin, expressed on fsLR",
)
plt.show()


parc_myelin = parcel_reduce(myelin, atlas_on_fsLR.path).values
parc_thickness = parcel_reduce(thickness, atlas_on_fsLR.path).values
print(f"{len(myelin)} vertices -> {len(parc_myelin)} regions")

plt.plot(parc_myelin, parc_thickness, "o")
plt.xlabel("myelin (parcellated)")
plt.ylabel("thickness (parcellated)")
plt.show()

In [ ]:
from neuromaps_prime.analysis.surfaces.nulls.nulls import (
    alexander_bloch,
    baum,
    hungarian,
    vasa,
)

sphere_path = graph.fetch_surface_atlas(
    space="fsLR", density="32k", hemisphere="right", resource_type="sphere"
).fetch()


for null_fn in (alexander_bloch, vasa, baum, hungarian):
    nl = null_fn(
        parc_myelin,
        surface=sphere_path,
        parcellation=str(atlas_on_fsLR),
        n_perm=500,
        seed=0,
    )
    res = compare_images(parc_myelin, parc_thickness, metric="pearsonr", nulls=nl)
    print(f"{null_fn.__name__:16s} p = {res.pvalue:.4f}")

---
## 7. Adding your own space *(offline)*

Now, let's say you have your own template, your own registration to a published space, or your own unpublished maps, and you want them to interoperate with everything already in the graph. You can add them to the Neuromaps-PRIME graph without any changes to the codebase.

For temporary modifications to the graph, you can use the Neuromaps-PRIME API. However, calls to the API to not make permanent changes to the underlying YAML files that define the spaces and transformations of the graph. Permenent changes, whether you want to keep these locally or distribute them publically, are made by modifying the actual YAML files. 

We will first explain how to use the API to edit the graph, then go into how the underlying YAML files are structured. 


### Anatomy of a space

Surfaces are keyed by vertex density, then surface type, then hemisphere. The `sphere` entry matters
most: surface registration happens on the sphere, so a space without spheres at a given density
cannot take part in surface transformations at that density. `white` and `pial` are needed for
projection between surface and volume.

In [ ]:
from pathlib import Path
from neuromaps_prime.graph import NeuromapsGraph
from neuromaps_prime.graph.models import Node, SurfaceAtlas

graph = NeuromapsGraph()

# A space is a Node; each surface it provides is a SurfaceAtlas. Every atlas
# declares the space, density, hemisphere and surface type it represents, so
# the graph can find it later without any filename parsing.
my_lab_template = Node(
    name="MyLabTemplate",
    species="macaque",
    description="Example laboratory template, version 1.0",
    references=["Author A, et al. 2026. Journal Name. doi:10.xxxx/xxxxx"],
    surfaces=[],
    volumes=[],
    surface_annotations=[],
    volume_annotations=[],
)

graph.add_node(my_lab_template.name, data=my_lab_template)

# add_atlas registers each surface on the node *and* in the lookup cache,
# which is what lets fetch_surface_atlas and the transformers find it.
for resource_type in ("sphere", "midthickness"):
    for hemi in ("left", "right"):
        graph.add_atlas(
            SurfaceAtlas(
                name=f"MyLabTemplate_32k_{hemi}_{resource_type}",
                description="Example laboratory template, version 1.0",
                file_path=Path(
                    f"/data/mylab/src-MyLabTemplate_den-32k"
                    f"_hemi-{hemi[0].upper()}_{resource_type}.surf.gii"
                ),
                space="MyLabTemplate",
                density="32k",
                hemisphere=hemi,
                resource_type=resource_type,
            )
        )

node = graph.get_node_data("MyLabTemplate")
print(f"{node.name} ({node.species}): {len(node.surfaces)} surfaces registered")
print(
    "reachable from fsLR yet?",
    graph.find_path("MyLabTemplate", "fsLR", edge_type="surface_to_surface")
    or "no — needs an edge",
)

A new space on its own is unreachable and only becomes useful once an edge connects it to a space already in
the graph.

### Exercise 3 — Connect your space

Declare a surface edge from `MyLabTemplate` to `Yerkes19` and register it, so that
`MyLabTemplate -> fsLR` resolves. The structure is:

```
from neuromaps_prime.graph.models import SurfaceTransform

graph.add_transform(
    SurfaceTransform(
        name=
        description=
        file_path=
        source_space=
        target_space=
        density=
        hemisphere=
        resource_type=
        provider=
    ),
    key=
)

```

In [ ]:
# YOUR CODE HERE
# my_edge = { ... }
# register(graph, surface_edges=[my_edge])

In [ ]:
from neuromaps_prime.graph.models import SurfaceTransform

for hemi in ("left", "right"):
    graph.add_transform(
        SurfaceTransform(
            name=f"MyLabTemplate_to_Yerkes19_32k_{hemi}",
            description="Registration produced in the workshop",
            file_path=Path(
                f"/data/xfm/src-MyLabTemplate_to-Yerkes19_den-32k"
                f"_hemi-{hemi[0].upper()}_sphere.surf.gii"
            ),
            source_space="MyLabTemplate",
            target_space="Yerkes19",
            density="32k",
            hemisphere=hemi,
            resource_type="sphere",
            provider="MyRegistration",
        ),
        key="surface_to_surface",
    )

print(graph.find_path("MyLabTemplate", "fsLR", edge_type="surface_to_surface"))

That is the payoff of the graph representation: one curated registration, correctly declared, yields
transformations to many spaces rather than one.

### A few easy mistakes to avoid

**Direction.** `from` and `to` define the direction data flow. Declare two edges if you have both
directions; a single edge produces exactly the one-way behaviour noted in Section 3.

In [ ]:
print(
    "MyLabTemplate -> fsLR:",
    graph.find_path("MyLabTemplate", "fsLR", edge_type="surface_to_surface")
    or "NO ROUTE",
)
print(
    "fsLR -> MyLabTemplate:",
    graph.find_path("fsLR", "MyLabTemplate", edge_type="surface_to_surface")
    or "NO ROUTE",
)

**Provenance naming.** The key beneath `surfaces` names the method that produced the registration.
Several methods may connect the same pair of spaces, and naming them lets a user choose between them.

**Silent failures.** A misspelled space name is not rejected: the builder creates a new node with
that name, so you gain a phantom node and a route that goes nowhere.

In [ ]:
from neuromaps_prime.graph.models import SurfaceTransform

g_demo = NeuromapsGraph()
before = set(g_demo.nodes(data=False))

typo_transform = SurfaceTransform(
    name="MyLabTemplate_to_Yerkes19_32k_left",
    description="Registration produced in the workshop",
    file_path=Path(
        "/data/xfm/src-MyLabTemplate_to-Yerkes19_den-32k_hemi-L_sphere.surf.gii"
    ),
    source_space="MyLabTemplate",
    target_space="Yerkes 19",  # note the space in "Yerkes 19"
    density="32k",
    hemisphere="left",
    resource_type="sphere",
    provider="MyRegistration",
)

g_demo.add_transform(typo_transform, key="surface_to_surface")

print("phantom nodes created:", sorted(set(g_demo.nodes(data=False)) - before))
print("node count:", len(before), "->", g_demo.number_of_nodes())
print(
    "MyLabTemplate -> fsLR:",
    g_demo.find_path("MyLabTemplate", "fsLR", edge_type="surface_to_surface")
    or "NO ROUTE",
)

# The phantom nodes are in the graph but carry no data at all
for name in sorted(set(g_demo.nodes(data=False)) - before):
    try:
        g_demo.get_node_data(name)
    except ValueError as e:
        print(f"  {name!r}: {e}")

### Making it permanent

Everything above lives only in this session. To persist it, write the same structures as YAML under
`resources/nodes/<species>/` and `resources/edges/surface/`. Annotations attach to a space under the
density at which they are defined:

```yaml
MyLabTemplate:
  surfaces:
    32k:
      annotation:
        MM:
          left:  /data/mylab/src-MyLabTemplate_den-32k_hemi-L_desc-MM_annot.func.gii
          right: /data/mylab/src-MyLabTemplate_den-32k_hemi-R_desc-MM_annot.func.gii
```

Name files so they declare their own provenance:

```
src-{space}_den-{density}_hemi-{hemi}_desc-{code}_annot.{ext}
```
We highly recommend following the BIDS naming convention, though this is not strictly enforced. 

To contribute back, fork the repository, add your YAML under `resources/`, confirm the graph loads
and the paths resolve, and open a pull request **documenting the provenance** of the registration:
the method that produced it and its original source. A registration is only interpretable if its
origin is documented. Is is assumed that the paths or URLs that are provided are publically available
and not, for instance, behind a data usage agreement. Data that is gated by a data usage agreement
can be integrated into Neuromaps-PRIME but requires that the user download the data separately before
connecting it to the graph. 

---
## 8. Benchmarking the software

**[needs a container and downloads]**

Read this before running it.

A round-trip benchmark transforms a map out along a route and back, then compares the returned map
with the original. It measures whether **the software is self-consistent**. It does not validate the
transformation:

- A **low** correlation flags something worth scrutiny: a density mismatch, medial-wall handling, or
  a poor registration. It is a starting point for your own validation experiments.
- A **high** correlation shows only self-consistency. A systematically *wrong* transformation
  composed with its inverse still returns the data to where it started.

In [ ]:
from neuromaps_prime.analysis import efficient_pearsonr

# Capture the result: it holds the real output location
rt_out = graph.surface_to_surface_transformer(
    transformer_type="metric",
    input_file=myelin_path,
    source_space="Yerkes19",
    target_space="fsLR",
    hemisphere="right",
    output_file_path="rt_out.func.gii",
)

rt_back = graph.surface_to_surface_transformer(
    transformer_type="metric",
    input_file=rt_out.path,  # <- the TransformResult, not a bare filename
    source_space="fsLR",
    target_space="Yerkes19",
    hemisphere="right",
    output_file_path="rt_back.func.gii",
)

original = nib.load(myelin_path).darrays[0].data
round_trip = nib.load(rt_back).darrays[0].data

r, _ = efficient_pearsonr(original, round_trip)
print(f"round-trip r = {float(r):.4f}")

Each resampling step introduces some discrepancy, so a longer route is generally less faithful than
a short one. `find_path` makes the number of steps explicit, and reporting it alongside results is
good practice.

---
## Where to go next

- Repository: https://github.com/childmindresearch/neuromaps-prime
- Add a template from your own lab (Section 7) and transform one of its maps into a human space
- Re-run Section 6 with a different null model and note whether your conclusion changes